# Robin transport and transient heat

The scalar RAD local form is skew in advection; its multiplier is $(-K\nabla u+\beta u/2)\cdot n$. The heat integrator uses backward Euler with time-dependent source and Dirichlet data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, solve_darcy, solve_brinkman, solve_elasticity


In [ ]:
from pymhm import solve_transport, solve_heat
mesh=TriangleMesh.unit_square(2)
space=SkeletonSpace(mesh,tuple(FaceSpace.uniform(1) for _ in mesh.faces))
def scalar(x):
    """Return the scalar affine spatial patch."""
    return 1+x[:,0]+2*x[:,1]
rad=solve_transport(mesh,velocity=(1.,.5),reaction=2.,source=lambda x:2*scalar(x)+2,dirichlet=scalar,skeleton=space)
assert rad.l2_error(scalar)<1e-10
times=[0.,.01,.03,.1]
solutions=solve_heat(mesh,times,initial=scalar,source=lambda x,t:np.ones(len(x)),dirichlet=lambda x,t:scalar(x)+t)
for time,result in zip(times[1:],solutions):
    error=result.l2_error(lambda x,t=time:scalar(x)+t)
    assert error<1e-10
    print('time',time,'space-time patch error',error)


Backward Euler is first order in time for general data. The affine space-time patch is exact and checks signs/history, but a temporal refinement experiment is still needed for nonaffine evolution (included in the automated tests).